In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)
COLLECTION_NAME = "vietnamese_docs_stage5_validation_20261017"

# # Xóa collection cũ
# client.delete_collection(collection_name=COLLECTION_NAME)
# print(f"✅ Đã xóa collection: {COLLECTION_NAME}")


✅ Đã xóa collection: vietnamese_docs_stage5_validation_20261017


In [2]:
# Lấy 1 point từ Qdrant
sample, _ = client.scroll(collection_name="vietnamese_docs", limit=1, with_payload=True)
pt = sample[0]
print(f"Qdrant point.id: {pt.id}")                 # VD: 42 hoặc "uuid"
print(f"payload['chunk_id']: {pt.payload.get('chunk_id')}")  # VD: "abc-def-uuid"
print(f"payload['doc_id']: {pt.payload.get('doc_id')}")

Qdrant point.id: 000993dd-b8fb-4971-ac86-bf133b899e09
payload['chunk_id']: 000993dd-b8fb-4971-ac86-bf133b899e09
payload['doc_id']: 8bd5c784-1456-4d27-b2d9-f5c24be320d4


In [4]:

from qdrant_client.http import models

# Tìm chunk có chứa "0.9597"
print("=" * 80)
print("TÌM CHUNK CHỨA '0.9597' (đáp án q032)\n")

scroll_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="text",
            match=models.MatchText(text="0.9597"),
        )
    ]
)
results, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    scroll_filter=scroll_filter,
    limit=10,
    with_payload=True,
)

if not results:
    print("❌ KHÔNG CÓ CHUNK NÀO chứa '0.9597' → Chunker/indexing bug!")
else:
    print(f"✅ Tìm thấy {len(results)} chunks chứa '0.9597':\n")
    for i, pt in enumerate(results, 1):
        text = pt.payload.get("text", "")
        has_epoch35 = "35" in text
        has_densenet = "DenseNet" in text or "densenet" in text.lower()
        print(f"--- Chunk {i} | id={str(pt.id)[:12]}... ---")
        print(f"  Có 'epoch 35': {has_epoch35}")
        print(f"  Có 'DenseNet': {has_densenet}")
        print(f"  Text preview: {text[:250]}")
        print()

TÌM CHUNK CHỨA '0.9597' (đáp án q032)

✅ Tìm thấy 1 chunks chứa '0.9597':

--- Chunk 1 | id=18af23ca-e65... ---
  Có 'epoch 35': True
  Có 'DenseNet': True
  Text preview: #### Kết quả huấn luyện và so sánh

Bảng 4: Bảng quá trình huấn luyện mô hình DenseNet201
Row 1: Epoch | Time | Acc | Loss | Val_acc | Val_loss | Learning rate
Row 2: 1 | 760s | 0.4467 | 1.2459 | 0.5068 | 1.9888 | 0.001
Row 3: 5 | 133s | 0.7062 | 0.5



In [2]:
import sys
from pathlib import Path
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.retrieval.hybrid_retriever import HybridRetriever


hr = HybridRetriever()
query = "Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?"

print("=" * 80)
print(f"QUERY: {query}\n")

# Dense only
print("--- DENSE TOP 5 ---")
dense = hr.vector_retriever.retrieve(query=query, top_k=10)
for c in dense[:5]:
    has_ans = "0.9597" in c.text
    marker = " ✅ CHỨA ĐÁP ÁN" if has_ans else ""
    print(f"  [score={c.score:.4f}] {c.text[:100]}{marker}")

print()
print("--- HYBRID TOP 5 ---")
hybrid = hr.retrieve(query=query, top_k=10)
for c in hybrid[:5]:
    has_ans = "0.9597" in c.text
    marker = " ✅ CHỨA ĐÁP ÁN" if has_ans else ""
    print(f"  [score={c.score:.4f}] {c.text[:100]}{marker}")

print()
print("--- SPARSE CHUNKS (từ hybrid) ---")
# Truy cập trực tiếp sparse để xem chunk nào được thêm vào
keywords = [w for w in query.split() if len(w) > 2]
print(f"Keywords: {keywords}")

2026-10-14 15:23:31 | INFO    | embedder:__init__:39 - Đang tải Embedding model: BAAI/bge-m3 trên thiết bị: cpu


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-10-14 15:23:41 | INFO    | embedder:__init__:49 - Tải model BAAI/bge-m3 thành công. Kích thước vector: 1024
QUERY: Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?

--- DENSE TOP 5 ---
2026-10-14 15:23:42 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_k=10)


The `sentences` argument was renamed and is now deprecated. Please use `inputs` instead.


2026-10-14 15:23:44 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
  [score=0.7074] Mô hình EfficientNetB0 cho thấy tốc độ hội tụ nhanh hơn DenseNet201, chỉ sau 10 epoch mà accuracy tr
  [score=0.6604] Mô hình kết thúc tại epoch 46 do cơ chế EarlyStopping với patience là 20, và kết quả đạt được cho th
  [score=0.6580] Trong khi đó, mô hình EfficientNetB0 đạt điểm tối ưu tại Epoch thứ 26 với độ chính xác trên tập kiểm
  [score=0.6407] *Bảng (thuộc: Kết quả huấn luyện và so sánh)*
Bảng 4: Bảng quá trình huấn luyện mô hình DenseNet201
 ✅ CHỨA ĐÁP ÁN
  [score=0.6318] *Bảng (thuộc: Kết quả huấn luyện và so sánh)*
Bảng 4: Bảng quá trình huấn luyện mô hình DenseNet201


--- HYBRID TOP 5 ---
2026-10-14 15:23:44 | INFO    | hybrid_retriever:retrieve:96 - Bắt đầu Hybrid Search: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (dense_w=0.6, sparse_w=0.4)
2026-10-14 15:23:44 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vect

In [5]:
# Lấy chunk chứa caption và chunk chứa data
sample, _ = client.scroll(
    collection_name="vietnamese_docs",
    scroll_filter=models.Filter(
        must=[models.FieldCondition(key="text", match=models.MatchText(text="Bảng quá trình huấn luyện mô hình DenseNet201"))]
    ),
    limit=5, with_payload=True,
)
for pt in sample:
    print(f"--- id={str(pt.id)[:12]}... ---")
    print(pt.payload.get("text", "")[:300])
    print()

--- id=20e36ac5-54c... ---
Bảng 2: Bảng mô tả tham số chính của mô hình DenseNet201

Giải thích lý do chọn kích thước ảnh đầu vào là 128 × 128 thay vì 224×224 như thiết kế ban đầu của mạng DenseNet201.

Việc lựa chọn như vây để giảm chi phí tính toán và rút ngắn thời gian huấn luyện. Qua quá trình thử nghiệm, nhóm nhận thấy c

--- id=491450cb-80d... ---
Bảng 4: Bảng quá trình huấn luyện mô hình DenseNet201

--- id=6dce8d44-a7a... ---
Bảng 3: Bảng mô tả tham số chính của mô hình EfficientNetB0 30

Bảng 4: Bảng quá trình huấn luyện mô hình DenseNet201 30

Bảng 5: Bảng quá trình huấn luyện mô hình EfficientNetB0 31

Bảng 6: Bảng mô tả chi tiết các độ đo đánh giá mô hình trên tập test 35

Bảng 7: Bảng so sánh các phương pháp xử lý ả



In [9]:
info = client.get_collection("vietnamese_docs")

print("Points:", info.points_count)
print("Dense vectors:", info.config.params.vectors)
print("Sparse vectors:", info.config.params.sparse_vectors)

Points: 760
Dense vectors: size=1024 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None
Sparse vectors: {'bm25': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}


In [8]:
client.create_vector_name(
    collection_name="vietnamese_docs",
    vector_name="bm25",
    vector_name_config=models.SparseVectorNameConfig(
        sparse=models.SparseVectorConfig(
            modifier=models.Modifier.IDF,
        ),
    ),
)

UpdateResult(operation_id=18, status=<UpdateStatus.COMPLETED: 'completed'>)

In [10]:

result = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=3,
    with_payload=True,
    with_vectors=False,
)

for point in result[0]:
    print("ID:", point.id)
    print("Payload:", point.payload)
    print("-" * 60)


ID: 003cdcec-73c5-4d6f-8a00-f9681de04a36
Payload: {'text': '| | duratio | n_in_gym_ | minutes | ) và mứ | c độ c | am kết | (subscription_model, |\n| --- | --- | --- | --- | --- | --- | --- | --- |\n| | sách ch | iết khấu và | dịch vụ | PT. Việ | c xây dự | ng mô hì | nh phân loại sẽ giúp |\n| | doanh | nghiệp cá n | hân hóa | ưu đãi c | ho đúng | nhóm kh | ách hàng có nguy cơ |\n| | rời bỏ, | thay vì áp d | ụng giả | m giá đại | trà gây t | ổn thất do | anh thu. |\n| | | | | | | | |\n| 2.5 | KHAI | THÁC DỮ | LIỆU | | | | |\n| | | | | | | | |', 'doc_id': '7a59e955-f125-4d9b-aab4-5874c2a09b92', 'chunk_id': '003cdcec-73c5-4d6f-8a00-f9681de04a36', 'chunk_index': 3, 'char_count': 481, 'word_count': 154, 'source': 'E:\\Personal Project\\vietnamese-rag\\data\\raw\\paper_data mining.pdf', 'file_name': 'paper_data mining.pdf', 'file_type': 'pdf', 'page_number': 30, 'total_pages': 71, 'section_title': 'ân tích hành vi tập luyện', 'breadcrumb': 'ân tích hành vi tập luyện', 'chunk_section_title':

In [11]:

from qdrant_client import models

collection = "vietnamese_docs"

points, _ = client.scroll(
    collection_name=collection,
    limit=1,
    with_payload=True,
    with_vectors=False,
)

if not points:
    raise RuntimeError("Collection khong co point nao.")

point = points[0]
text = point.payload.get("text", "").strip()

if not text:
    raise ValueError(f"Point {point.id} khong co text.")

client.update_vectors(
    collection_name=collection,
    points=[
        models.PointVectors(
            id=point.id,
            vector={
                "bm25": models.Document(
                    text=text,
                    model="Qdrant/bm25",
                )
            },
        )
    ],
    wait=True,
)

print("Da thu cap nhat BM25 cho point:", point.id)


Da thu cap nhat BM25 cho point: 003cdcec-73c5-4d6f-8a00-f9681de04a36


In [18]:

from qdrant_client import models

BATCH_SIZE = 64

BM25_OPTIONS = {
    "stemmer": {"type": "none"},
    "stopwords": {},
    "tokenizer": "multilingual",
}

offset = None
updated = 0

while True:
    points, next_offset = client.scroll(
        collection_name=COLLECTION_NAME,
        limit=BATCH_SIZE,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )

    if not points:
        break

    batch = []

    for point in points:
        text = (point.payload or {}).get("text")

        if not isinstance(text, str) or not text.strip():
            continue

        batch.append(
            models.PointVectors(
                id=point.id,
                vector={
                    "bm25": models.Document(
                        text=text,
                        model="Qdrant/bm25",
                        options=BM25_OPTIONS,
                    )
                },
            )
        )

    if batch:
        client.update_vectors(
            collection_name=COLLECTION_NAME,
            points=batch,
            wait=True,
        )
        updated += len(batch)
        print(f"Da cap nhat: {updated} points")

    if next_offset is None:
        break

    offset = next_offset

print("Hoan tat:", updated, "points")


Da cap nhat: 64 points
Da cap nhat: 128 points
Da cap nhat: 192 points
Da cap nhat: 256 points
Da cap nhat: 320 points
Da cap nhat: 384 points
Da cap nhat: 448 points
Da cap nhat: 512 points
Da cap nhat: 576 points
Da cap nhat: 640 points
Da cap nhat: 704 points
Da cap nhat: 760 points
Hoan tat: 760 points


In [20]:

from qdrant_client import models

query = "Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?"

response = client.query_points(
    collection_name="vietnamese_docs",
    query=models.Document(
        text=query,
        model="Qdrant/bm25",
        options=BM25_OPTIONS,
    ),
    using="bm25",
    limit=10,
    with_payload=True,
    with_vectors=False,
)

for i, point in enumerate(response.points, 1):
    payload = point.payload or {}

    print(f"\n--- Result {i} ---")
    print("Score:", round(point.score, 4))
    print("File:", payload.get("file_name"))
    print("Page:", payload.get("page_number"))
    print("Text:", payload.get("text", "")[:400])



--- Result 1 ---
Score: 30.0107
File: paper_Phân tích và xử lí ảnh.docx
Page: None
Text: Mô hình EfficientNetB0 cho thấy tốc độ hội tụ nhanh hơn DenseNet201, chỉ sau 10 epoch mà accuracy trên tập train đã đạt gần 90% nhưng val_loss vẫn còn khá cao. Ở epoch thứ 15 có thể thấy ReduceLROnPlateau đã kích hoạt để giảm tốc độ học xuống còn 1e-5 giúp val_loss bắt đầu ổn định và giảm sâu. Đến epoch 25 có thể thấy accuracy trên tập val đạt gần 90% và val_loss chỉ còn 0.3696 cho thấy mô hình đa

--- Result 2 ---
Score: 29.4541
File: paper_Phân tích và xử lí ảnh.docx
Page: None
Text: Mô hình kết thúc tại epoch 46 do cơ chế EarlyStopping với patience là 20, và kết quả đạt được cho thấy mô hình tốt nhất tại epoch thứ 26 với accuracy trên tập test là hơn 90% và val loss chỉ còn 0.3375. Có thể thấy hiệu suất và độ hội tụ của EfficientNetB0 thấp hơn DenseNet201.

--- Result 3 ---
Score: 23.6353
File: paper_Phân tích và xử lí ảnh.docx
Page: None
Text: Trong khi đó, mô hình EfficientNetB0 đạt điểm tối 

In [4]:

import sys
from pathlib import Path
# Thiết lập đường dẫn thư mục gốc dự án
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.retrieval.vector_retriever import VectorRetriever
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.pipeline import RetrievalPipeline
pipeline = RetrievalPipeline()

query = "Mô hình nào đạt kết quả tốt nhất cho bài toán phân loại khách hàng tiềm năng trong tập dữ liệu Shopping? "

# 1. Dense Retrieval
dense_results = pipeline.retrieve(
    query=query,
    top_k=50,
    top_n=5,
    use_hybrid=False,
    use_rerank=False,
)

# 2. Hybrid Retrieval
hybrid_results = pipeline.retrieve(
    query=query,
    top_k=50,
    top_n=5,
    use_hybrid=True,
    use_rerank=False,
)

# 3. Hybrid + Reranker
hybrid_rerank_results = pipeline.retrieve(
    query=query,
    top_k=50,
    top_n=5,
    use_hybrid=True,
    use_rerank=True,
)

for name, results in [
    ("Dense", dense_results),
    ("Hybrid", hybrid_results),
    ("Hybrid + Reranker", hybrid_rerank_results),
]:
    print(f"\n===== {name} =====")
    for chunk in results:
        print(
            chunk.rank,
            chunk.metadata.get("file_name"),
            chunk.metadata.get("page_number"),
            chunk.text[:200].replace("\n", " "),
        )

2026-10-14 17:12:07 | INFO    | embedder:__init__:39 - Đang tải Embedding model: BAAI/bge-m3 trên thiết bị: cpu


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-10-14 17:12:15 | INFO    | embedder:__init__:49 - Tải model BAAI/bge-m3 thành công. Kích thước vector: 1024
2026-10-14 17:12:16 | INFO    | reranker:__init__:37 - Đang tải Reranker model: BAAI/bge-reranker-v2-m3 trên thiết bị: cpu


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

2026-10-14 17:12:22 | INFO    | reranker:__init__:45 - Tải Reranker model BAAI/bge-reranker-v2-m3 thành công!
2026-10-14 17:12:22 | INFO    | retrieval_pipeline:retrieve:107 - Retrieval Pipeline | hybrid=False | rerank=False | candidate_k=50 | final_n=5
2026-10-14 17:12:22 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Mô hình nào đạt kết quả tốt nhất cho bài toán phân loại khách hàng tiềm năng trong tập dữ liệu Shopping?' (top_k=50)
2026-10-14 17:12:23 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 50 chunks tương đồng từ Qdrant.
2026-10-14 17:12:23 | INFO    | retrieval_pipeline:retrieve:128 - Candidate Retrieval tra ve 50 chunks.
2026-10-14 17:12:23 | INFO    | retrieval_pipeline:retrieve:163 - Tat Reranking; giu thu hang tu Retriever.
2026-10-14 17:12:23 | INFO    | retrieval_pipeline:retrieve:168 - Pipeline hoan tat: 5 chunks cuoi cung.
2026-10-14 17:12:23 | INFO    | retrieval_pipeline:retrieve:107 - Retrieval Pipeline | hybrid=True | rer

In [1]:
import json
from pathlib import Path
from qdrant_client import QdrantClient

COLLECTION = "vietnamese_docs_stage5_validation_20261017"

client = QdrantClient(
    host="localhost",
    port=6333,
)

info = client.get_collection(COLLECTION)

print("Dense vector schema:")
print(info.config.params.vectors)

print("Sparse vector schema:")
print(info.config.params.sparse_vectors)

def to_jsonable(value):
    """Chuyển dữ liệu Qdrant thành kiểu có thể ghi ra JSON."""
    if hasattr(value, "model_dump"):
        return to_jsonable(value.model_dump())

    if isinstance(value, dict):
        return {
            str(k): to_jsonable(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]

    if hasattr(value, "tolist"):
        return value.tolist()

    return value

output_path = Path("qdrant_export.jsonl")
offset = None
total = 0

with output_path.open("w", encoding="utf-8") as f:
    while True:
        points, next_offset = client.scroll(
            collection_name=COLLECTION,
            limit=100,
            offset=offset,
            with_payload=True,
            with_vectors=True,
        )

        for point in points:
            row = {
                "id": point.id,
                "payload": to_jsonable(point.payload or {}),
                "vector": to_jsonable(point.vector),
            }

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                    allow_nan=False,
                ) + "\n"
            )
            total += 1

        if next_offset is None:
            break

        offset = next_offset

print("Đã xuất:", total, "points")
print("File:", output_path.resolve())

Dense vector schema:
{'dense': VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)}
Sparse vector schema:
{'bm25': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=None, memory=None, datatype=None), modifier=<Modifier.IDF: 'idf'>)}
Đã xuất: 256 points
File: E:\Personal Project\vietnamese-rag\notebooks\qdrant_export.jsonl
